# Designing a star schema from business questions

Core stage, lesson C18. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** turn business questions into a grain, a fact table and dimensions with keys of their own, build the star in DuckDB, check its rows, its total and its keys, and answer a question through it. The last section is the project step of Part 3: the star schema of the cafes.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import decimal
import math
import numbers
import subprocess
import sys

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb

# Run this cell as it is. It also holds the answer checker and the helpers of the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3', 'Project step']


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(got, decimal.Decimal):
            got = int(got) if got == got.to_integral_value() else float(got)
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong


def show(sql, limit=12):
    """Run a statement and print its rows as text: the column names, then one line for each row."""
    cur = con.execute(sql)
    if cur.description is None:
        print("(the statement ran and returns no rows)")
        return
    rows = cur.fetchall()
    print(" | ".join(column[0] for column in cur.description))
    for row in rows[:limit]:
        print(" | ".join("NULL" if value is None else str(value) for value in row))
    if len(rows) > limit:
        print(f"... {len(rows)} rows in all")


def rows(sql):
    """Run a query and return its rows as lists, with decimals as numbers and dates as text."""
    cur = con.execute(sql)
    return [[float(v) if isinstance(v, decimal.Decimal) else (str(v) if hasattr(v, "isoformat") else v) for v in row] for row in cur.fetchall()]


def say_error(sql):
    """Run a statement; when the database refuses it, print the first line of its message instead of stopping."""
    try:
        show(sql)
    except duckdb.Error as error:
        print("Refused:", str(error).splitlines()[0])


def key_listing(table):
    """The primary and foreign keys of a table, as [type, column, position] lists (the query of unit 3 of C16)."""
    return [list(r) for r in con.execute(KEYS, [table]).fetchall()]


KEYS = """SELECT tc.constraint_type, kcu.column_name, kcu.ordinal_position AS position
FROM information_schema.table_constraints AS tc
JOIN information_schema.key_column_usage AS kcu
  ON kcu.constraint_name = tc.constraint_name AND kcu.table_name = tc.table_name
WHERE tc.table_name = ? AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
ORDER BY tc.constraint_type, kcu.ordinal_position, kcu.column_name"""

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/cafe-sales/v1/"
LOCAL = "../../datasets/cafe-sales/v1/"      # the same files, when this notebook runs inside a clone of the repository
TABLES = {                                   # the columns and types of datafromzero's cafe-sales.sql (no keys or constraints there)
    "stores": "'INTEGER','VARCHAR','VARCHAR','DATE'",
    "customers": "'INTEGER','DATE','VARCHAR','VARCHAR','INTEGER'",
    "products": "'INTEGER','VARCHAR','VARCHAR','DECIMAL(6,2)'",
    "orders": "'INTEGER','INTEGER','INTEGER','TIMESTAMP','VARCHAR','VARCHAR'",
    "order_items": "'INTEGER','INTEGER','INTEGER','INTEGER','DECIMAL(6,2)'",
}


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    import os
    import tempfile
    import urllib.request
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"


con = duckdb.connect()                       # a new, empty database every time this cell runs
for name, types in TABLES.items():
    con.execute(f"CREATE TABLE {name} AS FROM read_csv('{fetch(name)}', types=[{types}])")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
counts = [con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0] for name in TABLES]
assert counts == [3, 150, 16, 1800, 3287]
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## Unit 1: From questions to the grain

The owner's four questions: (1) which payment method do customers prefer at each cafe? (2) does the average order of a loyalty customer depend on the tier? (3) do weekends sell more coffee than weekdays? (4) how many units does an order hold at each cafe, on weekdays and on weekends?

**Predict first.** for each question, which number does it ask for and which words follow "by"? Then: should one row of the fact table stand for an order or for an order line? This query tests one of the two answers.

In [ ]:
show("""SELECT COUNT(*) AS orders,
       COUNT(*) FILTER (WHERE categories > 1) AS in_two_or_more_categories
FROM (SELECT i.order_id, COUNT(DISTINCT p.category) AS categories
      FROM order_items AS i
      JOIN products AS p ON p.product_id = i.product_id
      GROUP BY i.order_id)""")

## Unit 2: The keys of a dimension

This cell builds the dimensions of cafe, customer and day. The first two get their keys in order of name or id.

In [ ]:
con.execute("""CREATE OR REPLACE TABLE dim_store AS
SELECT CAST(row_number() OVER (ORDER BY store_name) AS INTEGER) AS store_key, store_id, store_name, city
FROM stores;

CREATE OR REPLACE TABLE dim_customer AS
SELECT 0 AS customer_key, CAST(NULL AS INTEGER) AS customer_id, 'unknown' AS loyalty_tier, 'unknown' AS city
UNION ALL
SELECT CAST(row_number() OVER (ORDER BY customer_id) AS INTEGER), customer_id, loyalty_tier, city
FROM customers;

CREATE OR REPLACE TABLE dim_date AS
SELECT CAST(strftime(d, '%Y%m%d') AS INTEGER) AS date_key, CAST(d AS DATE) AS full_date,
       month(d) AS month, dayname(d) AS weekday, isodow(d) >= 6 AS is_weekend
FROM generate_series(TIMESTAMP '2026-01-01', TIMESTAMP '2026-06-30', INTERVAL 1 DAY) AS t(d)""")
print("Three dimensions built.")

**Predict first.** the source has 150 customers and the days run from January 1 to June 30. How many rows does each dimension have?

In [ ]:
show("""SELECT 'dim_store' AS dimension, COUNT(*) AS rows, COUNT(DISTINCT store_key) AS distinct_keys FROM dim_store
UNION ALL SELECT 'dim_customer', COUNT(*), COUNT(DISTINCT customer_key) FROM dim_customer
UNION ALL SELECT 'dim_date', COUNT(*), COUNT(DISTINCT date_key) FROM dim_date
ORDER BY dimension""")

In [ ]:
show("SELECT * FROM dim_store ORDER BY store_key")

## Unit 3: The fact table, and the checks that come with it

This cell builds the fact table. `product_id` and `payment_method` stay as plain columns for now; the project step turns them into dimensions.

**Predict first.** the source has 3,287 lines worth 22,754.25 dollars. What does the fact table hold?

In [ ]:
con.execute("""CREATE OR REPLACE TABLE fact_sale AS
SELECT CAST(strftime(CAST(o.ordered_at AS DATE), '%Y%m%d') AS INTEGER) AS date_key,
       s.store_key,
       COALESCE(c.customer_key, 0) AS customer_key,
       i.product_id, o.payment_method,
       o.order_id, o.status AS order_status,
       i.quantity, i.discount_usd,
       i.quantity * p.unit_price_usd - i.discount_usd AS amount_usd
FROM order_items AS i
JOIN orders AS o ON o.order_id = i.order_id
JOIN products AS p ON p.product_id = i.product_id
JOIN dim_store AS s ON s.store_id = o.store_id
LEFT JOIN dim_customer AS c ON c.customer_id = o.customer_id""")
show("SELECT COUNT(*) AS rows, ROUND(SUM(amount_usd), 2) AS sales FROM fact_sale")

The third check: a `LEFT JOIN` from the fact table to a dimension returns an empty key where nothing matched.

In [ ]:
show("""SELECT COUNT(*) FILTER (WHERE d.date_key IS NULL) AS no_date,
       COUNT(*) FILTER (WHERE s.store_key IS NULL) AS no_store,
       COUNT(*) FILTER (WHERE c.customer_key IS NULL) AS no_customer
FROM fact_sale AS f
LEFT JOIN dim_date AS d ON d.date_key = f.date_key
LEFT JOIN dim_store AS s ON s.store_key = f.store_key
LEFT JOIN dim_customer AS c ON c.customer_key = f.customer_key""")

## Unit 4: Ask the questions of the star

**Predict first.** question 1, for the completed orders: which payment method comes first at each cafe?

In [ ]:
show("""SELECT s.store_name, f.payment_method, COUNT(DISTINCT f.order_id) AS orders
FROM fact_sale AS f
JOIN dim_store AS s ON s.store_key = f.store_key
WHERE f.order_status = 'completed'
GROUP BY s.store_name, f.payment_method
ORDER BY s.store_name, orders DESC, f.payment_method""")

## Finish it

### Exercise 1: Finish it

**Task:** Question 2: the average order of each tier. It needs the star of units 2 and 3. The starter counts the orders and adds up the sales for each tier, for every order. Finish it: keep only the completed orders and add the average amount of an order, the sales divided by the orders, rounded to 2 decimals as `avg_order`.

**Expected:** `answer_1` holds the rows of the finished query: 4 lists of [tier, orders, sales, avg_order] (bronze, gold, silver, unknown).

In [ ]:
answer_1 = rows("""SELECT c.loyalty_tier,
       COUNT(DISTINCT f.order_id) AS orders,
       ROUND(SUM(f.amount_usd), 2) AS sales
FROM fact_sale AS f
JOIN dim_customer AS c ON c.customer_key = f.customer_key
GROUP BY c.loyalty_tier
ORDER BY c.loyalty_tier""")      # finish the query: completed orders only, and the avg_order column

In [ ]:
_ = check("Exercise 1", "answer_1", [["bronze", 1044, 13312.5, 12.75], ["gold", 173, 2081.25, 12.03], ["silver", 581, 7348.5, 12.65], ["unknown", 1, 7.0, 7.0]],
          traps=[([["bronze", 1044, 13312.5, 12.75], ["gold", 173, 2081.25, 12.03], ["silver", 582, 7353.5, 12.63], ["unknown", 1, 7.0, 7.0]], "The canceled order 1461 is still in: filter on order_status = 'completed' before the GROUP BY.")])

<details>
<summary>Hint</summary>

Unit 4: the filter on `order_status` goes in a `WHERE` before the `GROUP BY`. The average of an order is a total divided by a count of distinct orders, not `AVG` of the lines.

</details>

<details>
<summary>Half-solution</summary>

```python
SELECT c.loyalty_tier, COUNT(DISTINCT f.order_id) AS orders, ROUND(SUM(f.amount_usd), 2) AS sales,
       ROUND(SUM(f.amount_usd) / ..., 2) AS avg_order
FROM fact_sale AS f
JOIN dim_customer AS c ON c.customer_key = f.customer_key
WHERE ...
```

</details>

<details>
<summary>Solution</summary>

```python
answer_1 = rows("""SELECT c.loyalty_tier,
       COUNT(DISTINCT f.order_id) AS orders,
       ROUND(SUM(f.amount_usd), 2) AS sales,
       ROUND(SUM(f.amount_usd) / COUNT(DISTINCT f.order_id), 2) AS avg_order
FROM fact_sale AS f
JOIN dim_customer AS c ON c.customer_key = f.customer_key
WHERE f.order_status = 'completed'
GROUP BY c.loyalty_tier
ORDER BY c.loyalty_tier""")
```

Bronze 12.75, gold 12.03, silver 12.65 and unknown 7.00 dollars. The tiers differ by at most 0.72 dollars, with the walk-in's one order of 7.00 apart. The filter matters for silver: the canceled order 1461 belongs to a silver customer, so the starter showed 582 orders and 7,353.50, and the completed orders are 581 and 7,348.50. The check cell reports the unfiltered rows.

</details>

## On your own

### Exercise 2: On your own

**Task:** Question 4: units in an order, by cafe and kind of day. Before you start, say what the result looks like: six rows, a cafe with weekday or weekend and a number of units. For the completed orders, divide the units by the orders (distinct order ids), rounded to 2 decimals as `units_per_order`. Write the code in the empty cell; it needs the star of units 2 and 3.

**Expected:** `answer_2` holds the rows of your query: 6 lists of [cafe, is_weekend, units_per_order].

In [ ]:
# Exercise 2: write your code here: answer_2 = rows(""" ... """)

In [ ]:
_ = check("Exercise 2", "answer_2", [["Campus", False, 2.79], ["Campus", True, 2.94], ["Harbor", False, 2.9], ["Harbor", True, 2.99], ["Old Town", False, 2.84], ["Old Town", True, 2.9]],
          traps=[([["Campus", False, 1.57], ["Campus", True, 1.57]], "That averages the quantity of a line. An order is a group of lines: divide the units by COUNT(DISTINCT order_id).")])

<details>
<summary>Hint</summary>

Join the fact table to `dim_store` and `dim_date` on their keys, filter on `order_status`, and group by the cafe and `is_weekend`. A unit count divided by a count of distinct orders.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"Six rows, each a cafe with weekday (false) or weekend (true), and a number between 2.7 and 3.0."*

```python
SELECT s.store_name, d.is_weekend, ROUND(SUM(f.quantity) * 1.0 / ..., 2) AS units_per_order
FROM fact_sale AS f
JOIN dim_store AS s ON s.store_key = f.store_key
JOIN dim_date AS d ON d.date_key = f.date_key
```

</details>

<details>
<summary>Solution</summary>

```python
answer_2 = rows("""SELECT s.store_name, d.is_weekend,
       ROUND(SUM(f.quantity) * 1.0 / COUNT(DISTINCT f.order_id), 2) AS units_per_order
FROM fact_sale AS f
JOIN dim_store AS s ON s.store_key = f.store_key
JOIN dim_date AS d ON d.date_key = f.date_key
WHERE f.order_status = 'completed'
GROUP BY s.store_name, d.is_weekend
ORDER BY s.store_name, d.is_weekend""")
```

Campus 2.79 on weekdays and 2.94 on weekends, Harbor 2.90 and 2.99, Old Town 2.84 and 2.90. Every cafe is a little higher on weekends, by between 0.06 and 0.15 units; each ratio rests on 140 to 470 orders. Question 4 needed no new table: the grain and the dimensions chosen from the other questions already covered it, which is the point of building from the questions.

</details>

## Check yourself

Without looking back: name the three checks of a star before anyone uses it. Then try the exercise.

<details>
<summary>answer</summary>

As many rows in the fact table as lines in the source, the same total, and no key in the fact table without a row in its dimension.

</details>

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague totals the units of each cafe from the star and reports that Harbor, which opened on April 1, sold the most. The checks of unit 3 pass on this star. Fix the join so that `answer_3` holds the right units of each cafe.

**Expected:** `answer_3` holds the rows of the fixed query: 3 lists of [cafe, units].

In [ ]:
answer_3 = rows("""SELECT s.store_name, SUM(f.quantity) AS units
FROM fact_sale AS f
JOIN dim_store AS s ON s.store_id = f.store_key
GROUP BY s.store_name
ORDER BY s.store_name""")      # colleague: the units of each cafe

In [ ]:
_ = check("Exercise 3", "answer_3", [["Campus", 2036], ["Harbor", 1041], ["Old Town", 2085]],
          traps=[([["Campus", 1041], ["Harbor", 2085], ["Old Town", 2036]], "Those are the right numbers on the wrong names. The ON matches the surrogate key of the fact table against the natural key of the dimension.")])

<details>
<summary>Hint</summary>

Compare the two sides of the `ON`. Which key does the fact table hold, and which column of `dim_store` is it matched against? Then add up the units of each store from `order_items` and `orders` and compare.

</details>

<details>
<summary>Half-solution</summary>

```python
JOIN dim_store AS s ON s.store_key = ...
```

</details>

<details>
<summary>Solution</summary>

```python
answer_3 = rows("""SELECT s.store_name, SUM(f.quantity) AS units
FROM fact_sale AS f
JOIN dim_store AS s ON s.store_key = f.store_key
GROUP BY s.store_name
ORDER BY s.store_name""")
```

The colleague matched the surrogate key of the fact table (`store_key`) against the natural key of the dimension (`store_id`). The numbers 1, 2 and 3 exist on both sides, so every row finds a partner, the join returns 3,287 rows and the total is right, but the cafes are mislabeled. The right answer is Campus 2,036 units, Harbor 1,041 and Old Town 2,085; the starter gave Campus 1,041, Harbor 2,085 and Old Town 2,036. A fourth check catches it: compare a total for each group with the source. The units of each cafe from `order_items` joined to `orders` and `stores` give the same three numbers as the fixed query. Row counts and grand totals cannot see a join that shuffles the labels.

</details>

## Project step: the star schema of the cafes

**The brief.** The owner wants the star finished and checked so that all four questions run on it. In the last section of the notebook, add `dim_product` and `dim_payment` the way unit 2 built the others (a surrogate key from `row_number()` in order of the product name and of the payment method, the natural key kept beside it), then rebuild `fact_sale` with five dimension keys and without the plain columns `product_id` and `payment_method`. Name these variables:

- `fact_rows`, the rows of `fact_sale`.
- `dim_rows`, a dictionary of the row counts of the five dimensions, with the keys `dim_store`, `dim_customer`, `dim_date`, `dim_product` and `dim_payment`.
- `unmatched`, the number of fact rows whose key finds no row in its dimension, added up over the five dimensions.
- `star_sales`, the sum of `amount_usd` of the fact table, rounded to 2 decimals.
- `gold_avg_order`, the average amount of an order for the gold customers' completed orders, rounded to 2 decimals.
- `coffee_weekend_pct`, the share of the completed coffee sales made on weekend days, in percent, rounded to 1 decimal.
- `weekend_day_pct`, the share of the 181 days that are weekend days, in percent, rounded to 1 decimal (a share of sales means little without the share of days).

Then write `design` with an f-string, so that every number in it comes from a variable. The check cell compares the variables with known answers; you read the design.

**Self-check list**, before you call it done:

1. The first line states the grain in one sentence.
2. It names the measures of the fact table and the five dimensions, with their row counts.
3. It says how the star was checked: the three checks of unit 3 (rows, total, unmatched keys) and a total for each group compared with the source (Exercise 3).
4. It names one question the star cannot answer and what would have to change.
5. Every number comes from a cell: none is typed.
6. It comes with a drawing of the finished star, on paper or in any tool: a box for the fact table with its grain written under the name, a box for each dimension, the keys marked, and "1" and "many" at the ends of each link. The drawing of unit 3 on the lesson page is the model; add the product and payment dimensions to it.

In [ ]:
# Project step: write your code here, then run the check cell below.

In [ ]:
_ = check_all("Project step", {"fact_rows": (3287, 0), "dim_rows": ({"dim_store": 3, "dim_customer": 151, "dim_date": 181, "dim_product": 16, "dim_payment": 3}, 0), "unmatched": (0, 0), "star_sales": (22754.25, 0.005), "gold_avg_order": (12.03, 0.005), "coffee_weekend_pct": (37.4, 0.05), "weekend_day_pct": (28.7, 0.05)})

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")